# A complete gentle-tree catalog and atlas

This example uses the field-bound algebra

$$0 \longrightarrow 1 \longrightarrow 2 \longleftarrow 3,$$

with the relation `a*b = 0`. It runs the dimension vector
`d = [1, 1, 1, 1]` through degree `3` over `F_2` and `F_5`.

The catalog theorem supplies all eight indecomposable string modules.
The atlas stores their ordered Ext table. Multiplicity enumeration
then finds every decomposition of `d` in catalog order.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from catalog_workflow import DEGREE_BOUND, TARGET_DIMENSIONS, render, run

with TemporaryDirectory() as directory:
    summary = run(Path(directory))
    render(summary)


## Catalog decomposition and Ext orientation

`catalog.entries[i]` is the certified module named by index `i`.
`atlas.ext_table.row(source, target)` stores `Ext^k(source, target)`
for `k = 0, ..., max_degree`. The source index comes first.


In [ ]:
item = summary["fields"][0]
catalog = item["catalog"]
atlas = item["atlas"]
solution = item["enumeration"].solutions[0]
row = atlas.ext_table.row(0, 0)
print("catalog entry dimensions:", [entry.dims for entry in catalog.entries])
print("first decomposition:", solution)
print(
    f"Ext orientation: source={row.source}, target={row.target}, "
    f"dimensions={row.dimensions}"
)
print("materialized dimensions:", atlas.materialize(solution).dims)


## Ordered Ext cells

The table labels rows by source and columns by target. Each cell shows
`(Ext^1, Ext^2)` and uses a dark background for a nonzero entry.


In [ ]:
from html import escape
from IPython.display import HTML, display

headers = "".join(
    f"<th>target {index}<br>{escape(str(entry.dims))}</th>"
    for index, entry in enumerate(catalog.entries)
)
table = [
    "<table><caption>Ordered Ext cells</caption>",
    "<tr><th>source / target</th>" + headers + "</tr>",
]
for source, source_entry in enumerate(catalog.entries):
    cells = [f"<th>source {source}<br>{escape(str(source_entry.dims))}</th>"]
    for target in range(len(catalog)):
        ext_one = atlas.ext_table.dim(source, target, 1)
        ext_two = atlas.ext_table.dim(source, target, 2)
        nonzero = ext_one or ext_two
        style = " style=\"background:#f4b183\"" if nonzero else ""
        cells.append(f"<td{style}>({ext_one}, {ext_two})</td>")
    table.append("<tr>" + "".join(cells) + "</tr>")
table.append("</table>")
display(HTML("".join(table)))


## The degree-two obstruction

The separated-simple module `S_0 \oplus S_2` has self-Ext row
`[2, 0, 1, 0]`. A degree-one-only check misses the nonzero
degree-two entry.


In [ ]:
assert TARGET_DIMENSIONS == [1, 1, 1, 1]
assert DEGREE_BOUND == 3
assert summary["obstruction"] == [2, 0, 1, 0]
print("degree-two obstruction:", summary["obstruction"][2])
